# CAMEL Model for all the cooperatives

## Functions

In [2]:
import pandas as pd

### Capital Adequacy

In [3]:
def get_value(excel_file, cuenta, company_name, date_):
    mask = (excel_file['CUENTA'] == cuenta) & (excel_file['fecha'] == date_)
    series = excel_file.loc[mask, company_name]
    if series.empty or series.isna().all():
        return None
    return series.iloc[-1]  # último corte del mes, no el primero

# Loss of Assets
def loss_of_assets(excel_file, company_name, year, month):
    date_ = pd.Timestamp(f'{int(year)}-{int(month)}-01')
    v1 = get_value(excel_file, 300000, company_name, date_)
    v2 = get_value(excel_file, 310000, company_name, date_)
    if v1 is None or v2 is None or v2 == 0:
        return None
    return v1 / v2

# Solvency Ratio
def solvency_ratio():
    return 0

# Indicator of the relationship between the non-reducible minimum social contributions and the Share Capital
def min_social_contrib_social_capital(excel_file, company_name, year, month):
    date_ = pd.Timestamp(f'{int(year)}-{int(month)}-01')
    
    mask1 = (excel_file['CUENTA'] == 311000) & (excel_file['fecha'] == date_)
    mask2 = (excel_file['CUENTA'] == 310000) & (excel_file['fecha'] == date_)
    
    account_coop1 = excel_file.loc[mask1, company_name]
    account_coop2 = excel_file.loc[mask2, company_name]

    if account_coop1.empty or account_coop2.empty:
        return 0
    
    value1 = list(account_coop1)[0]
    value2 = list(account_coop2)[0]
    
    return value1 / value2 if value2 != 0 else 0


# --- Sum the values of every row 'i' in a specific company
def summation(excel_file, index_list, company_name, year, month):
    suma = 0
    date_ = pd.Timestamp(f'{int(year)}-{int(month)}-01')
    for i in index_list:
        mask = (excel_file['CUENTA'] == i) & (excel_file['fecha'] == date_)
        account_coop = excel_file.loc[mask, company_name]
        if account_coop.empty or account_coop.isna().all():
            value = 0
        else:
            value = account_coop.iloc[0]  # o .sum() si esperas múltiples filas
        suma += value
    return suma


# Indicator of the relationship between Institutional Capital and Total Assets
def capital_contribution_ratio(excel_file, company_name, year, month):
    date_ = pd.Timestamp(f'{int(year)}-{int(month)}-01')
    capital_inst_indices = [320000, 330000, 340000]
    capital_inst = summation(excel_file, capital_inst_indices, company_name, year, month)
    
    mask_total_assets = (excel_file['CUENTA'] == 100000) & (excel_file['fecha'] == date_)
    total_assets = excel_file.loc[mask_total_assets, company_name]
    if total_assets.empty:
        return 0
    total_assets_ = list(total_assets)
    value1 = capital_inst
    value2 = total_assets_[0]

    return value1 / value2 if value2 != 0 else 0

### Assets Quality

In [4]:
# Functions for quality indicator by risk with penalties
def gross_portfolio(excel_file, company_name, year, month):
    gross_portfolio_indices = [
        140400, 140500, 141100, 141200, 144100, 144200,
        144800, 145400, 145500, 146100, 146200, 146900
    ]
    return summation(excel_file, gross_portfolio_indices, company_name, year, month)

def risk_quality_indicator(excel_file, company_name, year, month):
    risk_portfolio_indices = [
        140410, 140415, 140420, 140425, 140510, 140515, 140520, 140525,
        141110, 141115, 141120, 141125, 141210, 141215, 141220, 141225,
        144110, 144115, 144120, 144125, 144210, 144215, 144220, 144225,
        144810, 144815, 144820, 144825, 145410, 145415, 145420, 145425,
        145510, 145515, 145520, 145525, 146110, 146115, 146120, 146125,
        146210, 146215, 146220, 146225, 146910, 146915, 146920, 146925,
        146935, 146940, 146945, 146950
    ]
    qualified_portfolio = summation(excel_file, risk_portfolio_indices, company_name, year, month)
    gross_portf = gross_portfolio(excel_file,company_name, year, month)

    return qualified_portfolio / gross_portf if gross_portf != 0 else 0

def risk_quality_with_writeoffs(excel_file, company_name, year, month):
    # total qualified portfolio
    qualified_portfolio = risk_quality_indicator(excel_file, company_name, year, month)
    # writeoffs
    mask = (excel_file['CUENTA'] == 831015) & (excel_file['fecha'] == f'{int(year)}-{int(month)}-01')
    writeoffs_list = list(excel_file.loc[mask, company_name])
    writeoffs = writeoffs_list[0] if writeoffs_list else 0
    total_with_writeoffs_indices = [
        140400, 140500, 141100, 141200, 144100, 144200, 144800,
        145400, 145500, 146100, 146200, 146900, 831015
    ]
    total_with_writeoffs = summation(excel_file, total_with_writeoffs_indices, company_name, year, month)

    return (qualified_portfolio + writeoffs) / total_with_writeoffs if total_with_writeoffs != 0 else 0

# Total Portfolio at Risk Coverage Indicator
def total_risk_coverage_indicator(excel_file, company_name, year, month):
    deterioration_indices = [140800, 144500, 145100, 145800, 146500, 146800, 147100]
    provisions = summation(excel_file, deterioration_indices, company_name, year, month)
    gross_portf = gross_portfolio(excel_file, company_name, year, month)
    return provisions / gross_portf if gross_portf != 0 else 0

# Productive asset
def productive_assets_ratio(excel_file, company_name, year, month):
    productive_assets_indices = [
        112000, 120000, 130000, 140405, 140410, 140505, 140510,
        141105, 141110, 141205, 141210, 144105, 144110, 144205,
        144210, 144805, 144810, 145405, 145410, 145505, 145510,
        146105, 146110, 146205, 146210, 146905, 146910, 146930,
        146935, 160505, 161505
    ]
    productive_assets = summation(excel_file, productive_assets_indices, company_name, year, month)
    mask = (excel_file['CUENTA'] == 100000) & (excel_file['fecha'] == f'{int(year)}-{int(month)}-01')
    total_assets_list = list(excel_file.loc[mask, company_name])
    total_assets = total_assets_list[0] if total_assets_list else 0
    return productive_assets / total_assets if total_assets != 0 else 0

# Individual Coverage Indicator of the Unproductive Portfolio for the At-Risk Portfolio
def individual_coverage_nonproductive_portfolio(excel_file, company_name, year, month):
    provisions_cde_indices = [
        140815, 140820, 140825,
        144525, 144530, 144535, 144540, 144545, 144550,
        145115, 145120, 145125,
        145825, 145830, 145835, 145840, 145845, 145850,
        146525, 146530, 146535, 146540, 146545, 146550,
        147115, 147120, 147125, 147140, 147145, 147150
    ]
    provisions = summation(excel_file, provisions_cde_indices, company_name, year, month)

    overdue_portfolio_indices = [
        140415, 140420, 140425, 140515, 140520, 140525,
        141115, 141120, 141125, 141215, 141220, 141225,
        144115, 144120, 144125, 144215, 144220, 144225,
        144815, 144820, 144825, 145415, 145420, 145425,
        145515, 145520, 145525, 146115, 146120, 146125,
        146215, 146220, 146225, 146915, 146920, 146925,
        146940, 146945, 146950
    ]
    overdue = summation(excel_file, overdue_portfolio_indices, company_name, year, month)

    return provisions / overdue if overdue != 0 else 0

### Managerial Quality

In [5]:
# Operating Financial Margin Indicator
def financial_margin_operation(excel_file, company_name, year, month):
    date_filter = f'{int(year)}-{int(month)}-01'

    pos_list = list(excel_file.loc[(excel_file['CUENTA'] == 410000) & (excel_file['fecha'] == date_filter), company_name])
    positive_margin = pos_list[0] if pos_list else 0

    neg_list1 = list(excel_file.loc[(excel_file['CUENTA'] == 610000) & (excel_file['fecha'] == date_filter), company_name])
    negative_margin1 = neg_list1[0] if neg_list1 else 0

    serie = excel_file.loc[(excel_file['CUENTA'] == 700000) & (excel_file['fecha'] == date_filter), company_name]
    negative_margin2 = 0 if serie.empty else serie.iloc[0]

    sales_list = list(excel_file.loc[(excel_file['CUENTA'] == 410000) & (excel_file['fecha'] == date_filter), company_name])
    sales_income = sales_list[0] if sales_list else 0

    negative_margin = negative_margin1 + negative_margin2
    return (positive_margin - negative_margin) / sales_income if sales_income != 0 else 0


# Operating Margin Indicator (usa 'summation', no se modifica)
def operational_margin(excel_file, company_name, year, month):
    income_pos_indices = [410000, 422500]
    income_neg_indices = [610000, 700000, 510500, 510700, 511000, 511500, 540000]
    sales_indices = [410000, 422500]
    income_pos = summation(excel_file, income_pos_indices, company_name, year, month)
    income_neg = summation(excel_file, income_neg_indices, company_name, year, month)
    sales_income = summation(excel_file, sales_indices, company_name, year, month)
    return (income_pos - income_neg) / sales_income if sales_income != 0 else 0

# Indicator of the relationship between financial obligations and total liabilities
def financial_obligations_ratio(excel_file, company_name, year, month):
    date_filter = f'{int(year)}-{int(month)}-01'

    obligations_list = list(excel_file.loc[(excel_file['CUENTA'] == 230000) & (excel_file['fecha'] == date_filter), company_name])
    obligations = obligations_list[0] if obligations_list else 0

    total_liabilities_list = list(excel_file.loc[(excel_file['CUENTA'] == 200000) & (excel_file['fecha'] == date_filter), company_name])
    total_liabilities = total_liabilities_list[0] if total_liabilities_list else 0

    return obligations / total_liabilities if total_liabilities != 0 else 0

# Balance structure (usa 'summation', no se modifica)
def balance_structure(excel_file, company_name, year, month):
    productive_assets_indices = [
        112000, 120000, 130000, 140405, 140410, 140505, 140510,
        141105, 141110, 141205, 141210, 144105, 144110, 144205,
        144210, 144805, 144810, 145405, 145410, 145505, 145510,
        146105, 146110, 146205, 146210, 146905, 146910, 146930,
        146935, 160505, 161505
    ]
    interest_liabilities_indices = [210000, 230000]
    productive_assets = summation(excel_file, productive_assets_indices, company_name, year, month)
    interest_liabilities = summation(excel_file, interest_liabilities_indices, company_name, year, month)
    return productive_assets / interest_liabilities if interest_liabilities != 0 else 0


### Earnings Strength

In [6]:
# Return on equity indicator
def ROE(excel_file, company_name, year, month, n=1):
    fecha_actual = pd.Period(f"{year}-{month:02d}", freq="M")
    fecha_str = str(fecha_actual.to_timestamp().date())
    meses_previos = [str((fecha_actual - i).to_timestamp().date()) for i in range(1, 13)]

    num_values = excel_file.loc[
        (excel_file["CUENTA"] == 530000) & (excel_file["fecha"] == fecha_str),
        company_name
    ].values
    num = num_values[0] if len(num_values) > 0 else 0

    valores = excel_file.loc[
        (excel_file["CUENTA"] == 300000) & (excel_file["fecha"].isin(meses_previos)),
        company_name
    ].values

    if len(valores) == 0 or n == 0 or num == 0:
        return None

    prom_300000 = valores.mean()
    roe = (1 + ((num / prom_300000) / n))**12 - 1
    return roe


# Net margin indicator
def net_margin_indicator(excel_file, company_name, year, month):
    net_surplus_list = list(excel_file.loc[(excel_file['CUENTA'] == 530000) & (excel_file['fecha'] == f'{int(year)}-{int(month)}-01'), company_name])
    net_surplus = net_surplus_list[0] if net_surplus_list else 0
    incomes_indices = [410000, 422500]
    total_incomes = summation(excel_file, incomes_indices, company_name, year, month)
    return net_surplus / total_incomes if total_incomes != 0 else 0

# Indicator of return on invested capital
def ROIC(excel_file, company_name, year, month, n=1):
    fecha_actual = pd.Period(f"{year}-{month:02d}", freq="M")
    meses_previos = [str((fecha_actual - i).to_timestamp().date()) for i in range(1, 13)]

    # Numerator
    num_values = excel_file.loc[
        (excel_file["CUENTA"] == 530000) & (excel_file["fecha"] == str(fecha_actual.to_timestamp().date())),
        company_name
    ].values
    num = num_values[0] if len(num_values) > 0 else 0

    # Denominator
    cuentas = [210000, 230000, 300000]
    promedios = []
    for cuenta in cuentas:
        valores = excel_file.loc[
            (excel_file["CUENTA"] == cuenta) & (excel_file["fecha"].isin(meses_previos)),
            company_name
        ].values
        prom = valores.mean() if len(valores) > 0 else 0
        promedios.append(prom)

    denom = sum(promedios)

    if denom == 0 or n == 0 or num == 0:
        return None  # division by zero or no data

    roic = (1 + ((num / denom) / n))**12 - 1
    return roic

### Liquidicy Efficiency

In [7]:
# Liquidity Risk Indicator
def liquidity_risk():
    return 0

## Read the csv for all the cooperatives

In [8]:
coops = pd.read_csv("../tablas/Datos_2021_2025_cooperativas.csv")
coops['fecha'] = pd.to_datetime(coops['fecha'])
coops

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,COOPERATIVA AVP,FEBOR ENTIDAD COOPERATIVA,...,COOPERATIVA DE AHORRO Y CREDITO DE DROGUISTAS DETALLISTAS,COOPERATIVA DE AHORRO Y CREDITO COLANTA,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA DE AHORRO Y CREDITO CAJA UNION COOPERATIVA,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA,COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA,COOPERATIVA SUYA,fecha
0,100000,ACTIVO,1.453604e+11,3.387150e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,1.365023e+10,1.725947e+11,...,2.597647e+11,3.174706e+11,1.964690e+11,1.019668e+10,7.498950e+09,8.204825e+10,7.673873e+10,3.689148e+10,6.785967e+10,2022-02-01
1,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,1.094558e+10,1.566943e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,2.472604e+09,9.458176e+09,...,3.586048e+10,2.764945e+10,1.904275e+10,9.197822e+08,5.507006e+08,9.982434e+09,2.782317e+09,5.564935e+09,1.106163e+10,2022-02-01
2,110500,CAJA,2.868046e+08,8.229352e+08,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,6.956260e+08,3.535752e+08,...,2.000000e+06,5.735745e+09,2.193096e+09,6.007630e+07,4.744467e+08,4.490329e+08,2.018711e+08,1.615997e+08,1.612913e+09,2022-02-01
3,110505,CAJA GENERAL,2.791246e+08,8.180852e+08,3.589941e+08,1.718631e+08,1.146549e+08,0.000000e+00,6.956260e+08,3.535752e+08,...,0.000000e+00,5.732195e+09,2.182646e+09,5.907630e+07,4.738467e+08,4.450329e+08,1.998711e+08,1.600997e+08,1.612913e+09,2022-02-01
4,110510,CAJA MENOR,7.680000e+06,4.850000e+06,0.000000e+00,0.000000e+00,9.500000e+06,8.000000e+05,0.000000e+00,0.000000e+00,...,2.000000e+06,3.550000e+06,1.045000e+07,1.000000e+06,6.000000e+05,4.000000e+06,2.000000e+06,1.500000e+06,0.000000e+00,2022-02-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135686,931500,BIENES Y VALORES RECIBIDOS EN ADMINISTRACIÓN,0.000000e+00,2.470562e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.500000e+10,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2022-05-01
135687,931505,CARTERA FOGACOOP,0.000000e+00,2.470562e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.500000e+10,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2022-05-01
135688,960000,ACREEDORAS POR CONTRA (DB),7.498403e+09,4.308271e+11,1.432583e+11,6.690272e+09,6.667182e+10,2.998245e+10,2.233613e+10,3.804320e+10,...,4.587332e+11,3.204788e+11,1.410674e+11,1.611099e+10,6.967973e+09,1.638770e+10,7.480500e+08,0.000000e+00,1.997420e+10,2022-05-01
135689,960500,RESPONSABILIDADES CONTINGENTES POR EL CONTRARIO,7.498403e+09,4.308271e+11,1.432583e+11,6.690272e+09,6.667182e+10,2.998245e+10,2.233613e+10,3.804320e+10,...,4.587332e+11,3.204788e+11,1.410674e+11,1.611099e+10,6.967973e+09,1.638770e+10,7.480500e+08,0.000000e+00,1.997420e+10,2022-05-01


Generate new columns for the DataFrame

In [9]:
keys = list(coops.columns)
keys.remove("CUENTA")
keys.remove('NOMBRE CUENTA')
keys.remove('fecha')
print(keys)

['COOPERATIVA DE EMPLEADOS DE CAFAM', 'COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS', 'COOPERATIVA PARA EL BIENESTAR SOCIAL', 'COOPERATIVA FINANCIERA SAN FRANCISCO', 'COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA', 'COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA', 'COOPERATIVA AVP', 'FEBOR ENTIDAD COOPERATIVA', 'COOPERATIVA DE PROFESORES DE LA U NACIONAL DE COLOMBIA', 'CAJA COOPERATIVA CREDICOOP', 'COOPERATIVA DE AHORRO Y CREDITO DE SURAMERICA', 'FINANCIERA COOPERATIVA COLOMBIANA DE INGENIEROS', 'COOPERATIVA DE AHORRO Y CREDITO DE TRABAJADORES DE PELDAR Y OTROS DE COLOMBIA', 'COOPERATIVA ALIANZA', 'COOPERATIVA DEL MAGISTERIO', 'COOPERATIVA DE AHORRO Y CREDITO CREDIFLORES', 'COOPERATIVA DE AHORRO Y CREDITO UNIVERSIDAD SANTO TOMAS', 'CAJA COOPERATIVA PETROLERA', 'COOPERATIVA TEXAS LTDA', 'COOPERATIVA DE LOS TRABAJADORES DEL INSTITUTO DE LOS SEGUROS SOCIALES', 'COOPERATIVA DE TRABAJADORES DE BAVARIA DIRECCION Y VENTAS LTDA', 'COPROCENVA COOPERATIVA DE AHORRO Y CREDITO', 'COO

In [10]:
indexes_camel = ['Quebranto Patrimonial', 'Relación Solvencia', 'Relación entre Aportes sociales mínimos no reducibles y Capital Social', 'Relación entre el Capital Institucional y el Activo Total', 
                 'Indicador de calidad por riesgo', 'Indicador de calidad por riesgo con castigos', 'Indicador de Cobertura de la Cartera Total en Riesgo', 'Activo Productivo', 'Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo',
                 'Indicador de Margen Financiero de Operación', 'Indicador de Margen Operacional', 'Indicador de relación entre las obligaciones financieras y el pasivo total', 'Estructura de Balance',
                 'Indicador de rentabilidad sobre recursos propios - ROE', 'Indicador de margen neto', 'Indicador de rentabilidad sobre el capital invertido - ROIC',
                 'Indicador de Riesgo de Liquidez - IRL']
years = range(2021, 2026)
months = range(1, 13)

We need to parallelize the next cells, because there is a lot of calculations there (for the big dataset)

### Capital

In [11]:
from concurrent.futures import ThreadPoolExecutor
from functools import partial

In [17]:
def calculate_values_capital(company, coops, years, months, indices_camel):
    rows = []
    for y in years:
        for m in months:
            c1 = loss_of_assets(coops, company, y, m)
            c2 = solvency_ratio()
            c3 = min_social_contrib_social_capital(coops, company, y, m)
            c4 = capital_contribution_ratio(coops, company, y, m)

            rows.append([y, m, indices_camel[0], company, c1])
            rows.append([y, m, indices_camel[1], company, c2])
            rows.append([y, m, indices_camel[2], company, c3])
            rows.append([y, m, indices_camel[3], company, c4])
    return rows

# parameters
partial_func_c = partial(calculate_values_capital, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# execute for every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_c, keys))

# organize all the info in a dataframe
rows_c = [fila for sublista in resultados for fila in sublista]

df = pd.DataFrame(rows_c, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot = df.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot.columns.name = None

### Assets

In [28]:
def calculate_values_assets_1(company, coops, years, months, indices_camel):
    rows = []
    for y in years:
        for m in months:
            a1 = risk_quality_indicator(coops, company, y, m)
            a2 = risk_quality_with_writeoffs(coops, company, y, m)
            a3 = total_risk_coverage_indicator(coops, company, y, m)

            rows.append([y, m, indices_camel[4], company, a1])
            rows.append([y, m, indices_camel[5], company, a2])
            rows.append([y, m, indices_camel[6], company, a3])
    return rows


def calculate_values_assets_2(company, coops, years, months, indices_camel):
    rows = []
    for y in years:
        for m in months:
            # values for A: ASSETS
            a4 = productive_assets_ratio(coops, company, y, m)
            a5 = individual_coverage_nonproductive_portfolio(coops, company, y, m)

            rows.append([y, m, indices_camel[7], company, a4])
            rows.append([y, m, indices_camel[8], company, a5])
    return rows      

first part

In [29]:
partial_func_a = partial(calculate_values_assets_1, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_a, keys))

rows_a = [fila for sublista in resultados for fila in sublista]

# Filtra filas donde el valor CAMEL no es nulo/NaN y el año es 2025
rows_a_filtrados = [fila for fila in rows_a if fila[0] == 2025 and pd.notnull(fila[4]) and fila[4] != 0]

# incluir todos los años pero omitir cooperativas sin valores en 2025:
# Primero identifica las cooperativas válidas
coops_con_valores_2025 = set(fila[3] for fila in rows_a if fila[0] == 2025 and pd.notnull(fila[4]) and fila[4] != 0)
rows_a_filtrados = [fila for fila in rows_a if fila[3] in coops_con_valores_2025]

df_a = pd.DataFrame(rows_a_filtrados, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_a = df_a.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_a.columns.name = None


In [30]:
df_pivot_a

,Año,Mes,índice CAMEL,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,CAJA COOPERATIVA CREDICOOP,CAJA COOPERATIVA PETROLERA,CASA NACIONAL DEL PROFESOR,CESCA COOPERATIVA DE AHORRO Y CREDITO,COOCERVUNION COOPERATIVA DE AHORRO Y CREDITO,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,...,COPERATIVA MULTIACTIVA DE EDUCADORES DE BOYACA,COPROCENVA COOPERATIVA DE AHORRO Y CREDITO,EMPRESA COOPERATIVA DE AHORRO Y CREDITO SIGLO XX LTDA.,FEBOR ENTIDAD COOPERATIVA,FINANCIERA COOPERATIVA COLOMBIANA DE INGENIEROS,FORJAR COOPERATIVA DE AHORRO Y CREDITO,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURSOS NATURALES,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,"MULTIACTIVA EL ROBLE, ENTIDAD COOPERATIVA"
0,2021,1,Indicador de Cobertura de la Cartera Total en ...,0.055566,0.053379,0.115744,0.022701,0.049609,0.010610,0.087994,...,0.019292,0.026337,0.034718,0.031671,0.063798,0.055208,0.068745,0.015832,0.034574,0.030314
1,2021,1,Indicador de calidad por riesgo,0.029344,0.086836,0.124619,0.018043,0.063940,0.014136,0.103050,...,0.019040,0.029836,0.057121,0.071968,0.047502,0.060850,0.041153,0.015511,0.061412,0.030185
2,2021,1,Indicador de calidad por riesgo con castigos,0.019615,0.095331,0.075134,0.003303,0.033969,0.018697,0.039720,...,0.000516,0.012715,0.073079,0.007629,0.231033,0.035990,0.007803,0.011921,0.141528,0.004795
3,2021,2,Indicador de Cobertura de la Cartera Total en ...,0.065230,0.038193,0.089557,0.020699,0.046888,0.017331,0.085480,...,0.025193,0.028450,0.039215,0.040037,0.070771,0.035189,0.066121,0.017910,0.052338,0.032886
4,2021,2,Indicador de calidad por riesgo,0.040190,0.075898,0.089526,0.016155,0.045786,0.150525,0.119530,...,0.046458,0.031505,0.056031,0.080705,0.026147,0.066102,0.051620,0.037568,0.073247,0.048107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,2025,11,Indicador de calidad por riesgo,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
176,2025,11,Indicador de calidad por riesgo con castigos,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
177,2025,12,Indicador de Cobertura de la Cartera Total en ...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
178,2025,12,Indicador de calidad por riesgo,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


second part

In [31]:
partial_func_a_2 = partial(calculate_values_assets_2, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados_2 = list(executor.map(partial_func_a_2, keys))

rows_a_2 = [fila for sublista in resultados_2 for fila in sublista]

# incluir todos los años pero omitir cooperativas sin valores en 2025:
# Primero identifica las cooperativas válidas
coops_con_valores_2025_2 = set(fila[3] for fila in rows_a_2 if fila[0] == 2025 and pd.notnull(fila[4]) and fila[4] != 0)
rows_a_filtrados_2 = [fila for fila in rows_a_2 if fila[3] in coops_con_valores_2025_2]

df_a_2 = pd.DataFrame(rows_a_filtrados_2, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_a_2 = df_a_2.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_a_2.columns.name = None

In [32]:
df_pivot_a_2

,Año,Mes,índice CAMEL,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,CAJA COOPERATIVA CREDICOOP,CAJA COOPERATIVA PETROLERA,CASA NACIONAL DEL PROFESOR,CESCA COOPERATIVA DE AHORRO Y CREDITO,COOCERVUNION COOPERATIVA DE AHORRO Y CREDITO,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,...,COPERATIVA MULTIACTIVA DE EDUCADORES DE BOYACA,COPROCENVA COOPERATIVA DE AHORRO Y CREDITO,EMPRESA COOPERATIVA DE AHORRO Y CREDITO SIGLO XX LTDA.,FEBOR ENTIDAD COOPERATIVA,FINANCIERA COOPERATIVA COLOMBIANA DE INGENIEROS,FORJAR COOPERATIVA DE AHORRO Y CREDITO,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURSOS NATURALES,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,"MULTIACTIVA EL ROBLE, ENTIDAD COOPERATIVA"
0,2021,1,Activo Productivo,0.849683,0.836182,0.847524,0.787342,0.758971,0.813569,0.862547,...,0.839576,0.927624,0.793425,0.874941,0.597944,0.767561,0.946596,0.901930,0.803447,0.848908
1,2021,1,Indicador de Cobertura individual de la carter...,0.368052,0.637977,0.820234,0.914385,0.787072,0.217691,0.888338,...,0.584411,0.723133,0.706156,0.419189,0.181554,0.761799,0.532663,0.206664,0.613325,0.852915
2,2021,2,Activo Productivo,0.891693,0.856016,0.868504,0.806438,0.746393,0.789267,0.893226,...,0.873551,0.949672,0.796180,0.900145,0.664628,0.734324,0.957687,0.923268,0.885845,0.806865
3,2021,2,Indicador de Cobertura individual de la carter...,0.630189,0.443646,0.823900,0.852717,0.807198,0.137426,0.850376,...,0.457286,0.702165,0.699594,0.471055,0.436296,0.179965,0.413926,0.222500,0.666058,0.487356
4,2021,3,Activo Productivo,0.845382,0.843916,0.852847,0.794876,0.762370,0.810318,0.870493,...,0.844909,0.926671,0.798224,0.886912,0.565216,0.760659,0.950504,0.898259,0.793209,0.847713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2025,10,Indicador de Cobertura individual de la carter...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
116,2025,11,Activo Productivo,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
117,2025,11,Indicador de Cobertura individual de la carter...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
118,2025,12,Activo Productivo,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Managerial

In [33]:
def calculate_values_managerial(company, coops, years, months, indices_camel):
    rows = []
    for y in years:
        for m in months:
            # values for M: MANAGERIAL
            m1 = financial_margin_operation(coops, company, y, m)
            m2 = operational_margin(coops, company, y, m)
            m3 = financial_obligations_ratio(coops, company, y, m)
            m4 = balance_structure(coops, company, y, m)

            rows.append([y, m, indices_camel[9], company, m1])
            rows.append([y, m, indices_camel[10], company, m2])
            rows.append([y, m, indices_camel[11], company, m3])
            rows.append([y, m, indices_camel[12], company, m4])
    return rows


In [34]:
partial_func_m = partial(calculate_values_managerial, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_m, keys))

rows_m = [fila for sublista in resultados for fila in sublista]

df_m = pd.DataFrame(rows_m, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_m = df_m.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_m.columns.name = None

In [35]:
df_pivot_m

,Año,Mes,índice CAMEL,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,CAJA COOPERATIVA CREDICOOP,CAJA COOPERATIVA PETROLERA,CASA NACIONAL DEL PROFESOR,CESCA COOPERATIVA DE AHORRO Y CREDITO,COOCERVUNION COOPERATIVA DE AHORRO Y CREDITO,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,...,COPERATIVA MULTIACTIVA DE EDUCADORES DE BOYACA,COPROCENVA COOPERATIVA DE AHORRO Y CREDITO,EMPRESA COOPERATIVA DE AHORRO Y CREDITO SIGLO XX LTDA.,FEBOR ENTIDAD COOPERATIVA,FINANCIERA COOPERATIVA COLOMBIANA DE INGENIEROS,FORJAR COOPERATIVA DE AHORRO Y CREDITO,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURSOS NATURALES,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,"MULTIACTIVA EL ROBLE, ENTIDAD COOPERATIVA"
0,2021,1,Estructura de Balance,1.738541,1.687269,1.515012,1.187705,1.964584,1.369032,1.338180,...,1.228318,1.335712,1.808357,1.287252,1.364070,1.475662,2.775355,2.644433,1.878060,1.863718
1,2021,1,Indicador de Margen Financiero de Operación,0.835447,0.801455,0.709589,0.703449,0.833326,0.637803,0.651795,...,0.652784,0.747763,0.867811,0.642544,0.718867,0.860370,0.876935,0.911610,0.885306,0.866134
2,2021,1,Indicador de Margen Operacional,0.281662,-0.001508,-0.044339,0.292152,0.219683,0.098861,0.147045,...,0.230574,0.127692,0.136075,0.192058,-0.105607,0.267655,0.281239,0.400028,0.020771,0.167413
3,2021,1,Indicador de relación entre las obligaciones f...,0.000000,0.033832,0.000000,0.060400,0.000000,0.000000,0.000000,...,0.000000,0.018394,0.065830,0.000000,0.000000,0.000000,0.000000,0.297758,0.122385,0.000000
4,2021,2,Estructura de Balance,1.807411,1.820013,1.496788,1.192809,1.889901,1.364444,1.389044,...,1.270608,1.354679,1.777400,1.284604,1.482266,1.321589,2.667183,2.199104,2.206180,1.800508
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,2025,11,Indicador de relación entre las obligaciones f...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
236,2025,12,Estructura de Balance,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
237,2025,12,Indicador de Margen Financiero de Operación,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
238,2025,12,Indicador de Margen Operacional,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Earnings

In [36]:
def calculate_values_earnings(company, coops, years, months, indices_camel):
    rows = []
    for y in years:
        for m in months:
            # values for E: EARNINGS
            e1 = ROE(coops, company, y, m)
            e2 = net_margin_indicator(coops, company, y, m)
            e3 = ROIC(coops, company, y, m)

            rows.append([y, m, indices_camel[13], company, e1])
            rows.append([y, m, indices_camel[14], company, e2])
            rows.append([y, m, indices_camel[15], company, e3])
    return rows

In [38]:
partial_func_e = partial(calculate_values_earnings, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_e, keys))

rows_e = [fila for sublista in resultados for fila in sublista]

df_e = pd.DataFrame(rows_e, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_e = df_e.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_e.columns.name = None

In [39]:
df_pivot_e

,Año,Mes,índice CAMEL,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,CAJA COOPERATIVA CREDICOOP,CAJA COOPERATIVA PETROLERA,CASA NACIONAL DEL PROFESOR,CESCA COOPERATIVA DE AHORRO Y CREDITO,COOCERVUNION COOPERATIVA DE AHORRO Y CREDITO,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,...,COPERATIVA MULTIACTIVA DE EDUCADORES DE BOYACA,COPROCENVA COOPERATIVA DE AHORRO Y CREDITO,EMPRESA COOPERATIVA DE AHORRO Y CREDITO SIGLO XX LTDA.,FEBOR ENTIDAD COOPERATIVA,FINANCIERA COOPERATIVA COLOMBIANA DE INGENIEROS,FORJAR COOPERATIVA DE AHORRO Y CREDITO,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURSOS NATURALES,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,"MULTIACTIVA EL ROBLE, ENTIDAD COOPERATIVA"
0,2021,1,Indicador de margen neto,0.251205,-0.019653,-0.079921,0.242726,0.212640,0.133493,0.132738,...,0.250731,0.135008,0.118645,0.205857,0.137903,0.299471,0.406515,0.405155,0.043917,0.164710
1,2021,2,Indicador de margen neto,0.141883,0.044622,0.051629,0.047383,0.076787,-0.256753,0.096169,...,0.070200,0.055364,0.074456,0.140927,0.150408,0.312514,0.319443,0.257903,0.275352,0.070679
2,2021,3,Indicador de margen neto,0.244838,0.015718,0.005008,0.078838,0.215565,0.097741,0.163545,...,0.118091,0.150393,0.005710,0.158648,0.178791,0.331536,0.306543,0.311750,0.207625,0.127797
3,2021,4,Indicador de margen neto,0.263801,0.000586,-0.013719,0.066885,0.183911,0.099448,0.182034,...,0.116497,0.139552,0.068146,0.149240,0.213328,0.338229,0.283060,0.301787,0.247468,0.137001
4,2021,5,Indicador de margen neto,0.255216,0.052473,0.028472,0.085381,0.144892,0.096910,0.133410,...,0.070819,0.085680,0.096430,0.139214,0.190226,0.349099,0.312128,0.255291,0.257131,0.102592
5,2021,6,Indicador de margen neto,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,2021,7,Indicador de margen neto,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,2021,8,Indicador de margen neto,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,2021,9,Indicador de margen neto,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,2021,10,Indicador de margen neto,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


### Liquidicy

In [40]:
def calculate_values_liquidicy(company, coops, years, months, indices_camel):
    rows = []
    for y in years:
        for m in months:
            # values for L: LIQUIDICY
            l1 = liquidity_risk()
            rows.append([y, m, indices_camel[16], company, l1])
    return rows

In [41]:
partial_func_l = partial(calculate_values_liquidicy, coops=coops, years=years, months=months, indices_camel=indexes_camel)

# every company
with ThreadPoolExecutor() as executor:
    resultados = list(executor.map(partial_func_l, keys))

rows_l = [fila for sublista in resultados for fila in sublista]

df_l = pd.DataFrame(rows_l, columns=['Año', 'Mes', 'índice CAMEL', 'Cooperativa', 'valor CAMEL'])
df_pivot_l = df_l.pivot_table(index=['Año', 'Mes', 'índice CAMEL'], columns='Cooperativa', values='valor CAMEL').reset_index()
df_pivot_l.columns.name = None

In [42]:
df_pivot_l

,Año,Mes,índice CAMEL,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,CAJA COOPERATIVA CREDICOOP,CAJA COOPERATIVA PETROLERA,CASA NACIONAL DEL PROFESOR,CESCA COOPERATIVA DE AHORRO Y CREDITO,COOCERVUNION COOPERATIVA DE AHORRO Y CREDITO,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,...,COPERATIVA MULTIACTIVA DE EDUCADORES DE BOYACA,COPROCENVA COOPERATIVA DE AHORRO Y CREDITO,EMPRESA COOPERATIVA DE AHORRO Y CREDITO SIGLO XX LTDA.,FEBOR ENTIDAD COOPERATIVA,FINANCIERA COOPERATIVA COLOMBIANA DE INGENIEROS,FORJAR COOPERATIVA DE AHORRO Y CREDITO,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURSOS NATURALES,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,"MULTIACTIVA EL ROBLE, ENTIDAD COOPERATIVA"
0,2021,1,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2021,2,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021,3,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2021,4,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021,5,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,2021,6,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,2021,7,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,2021,8,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,2021,9,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,2021,10,Indicador de Riesgo de Liquidez - IRL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## CSV for BD

Write on .csv the data from DataFrames:

DataFrames's names and their title:
* 'df_pivot': Capital
* 'df_pivot_a' y 'df_pivot_a_2': Assets
* 'df_pivot_m': Managerial
* 'df_pivot_e': Earnings
* 'df_pivot_l': Liquidicy

In [43]:
df_pivot.to_csv("../tablas/camel/Capital.csv", index=False, encoding="utf-8")
# assets
assets = pd.concat([df_pivot_a, df_pivot_a_2], ignore_index=True)
assets.to_csv("../tablas/camel/Assets.csv", index=False, encoding="utf-8")
# ---
df_pivot_m.to_csv("../tablas/camel/Managerial.csv", index=False, encoding="utf-8")
df_pivot_e.to_csv("../tablas/camel/Earnings.csv", index=False, encoding="utf-8")
df_pivot_l.to_csv("../tablas/camel/Liquidity.csv", index=False, encoding="utf-8")